# Train OCR text Detector quick example

In [1]:
import os
import sys
import torch
import warnings
from datetime import datetime
from matplotlib import pyplot as plt
warnings.filterwarnings('ignore')

# change this property
NOMEROFF_NET_DIR = os.path.abspath('../../../../')


sys.path.append(NOMEROFF_NET_DIR)

from nomeroff_net.pipes.number_plate_text_readers.base.ocr import OCR

In [2]:
plt.rcParams["figure.figsize"] = (10, 10)

In [3]:
%matplotlib inline 

In [8]:
# auto download latest dataset
from nomeroff_net.tools import modelhub

# auto download latest dataset
#info = modelhub.download_dataset_for_model("EuUaFrom2004")
#PATH_TO_DATASET = info["dataset_path"]

# local path dataset
#PATH_TO_DATASET = os.path.join(NOMEROFF_NET_DIR, "./data/dataset/TextDetector/ocr_example")
PATH_TO_DATASET = "/mnt/datasets/nomeroff-net/autoriaNumberplateOcrUa-2024-12-20"

color_channels, height, width = 3, 100, 400
linear_size = 512 # 384, 416, 448, 480, 512

In [9]:
DATASET_NAME = "eu_2004_2015"
VERSION = f"{datetime.now().strftime('%Y_%m_%d')}_pytorch_lightning"

RESULT_MODEL_PATH = os.path.join(NOMEROFF_NET_DIR, 
                                 "models/", 
                                 'anpr_ocr_{}_{}.ckpt'.format(DATASET_NAME, VERSION))

In [10]:
RESULT_MODEL_PATH

'/mnt/raid/var/www/nomeroff-net/models/anpr_ocr_eu_2004_2015_2024_12_20_pytorch_lightning.ckpt'

In [11]:
class eu_ua_2004_2015(OCR):
    def __init__(self):
        OCR.__init__(self)
        # only for usage model
        # in train generate automaticly
        self.letters = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "A", "B", "C", "E", "H", "I", "K", "L",
                        "M", "O", "P", "T", "X", "Z"]


        self.height = 100
        self.width = 400
        self.hidden_size = 32
        self.linear_size = linear_size
        
        # Train hyperparameters
        self.batch_size = 16
        self.epochs = 5
        self.gpus = torch.cuda.device_count()

In [12]:
ocrTextDetector = eu_ua_2004_2015()
model = ocrTextDetector.prepare(PATH_TO_DATASET, use_aug=False, num_workers=1)

GET ALPHABET
Max plate length in "val": 8
Max plate length in "train": 8
Max plate length in "test": 8
Letters train  {'H', '7', '0', '9', 'L', '5', 'X', 'Y', '3', '2', 'E', 'D', 'M', 'B', 'A', '1', 'O', '4', 'K', 'G', 'C', '8', 'P', 'Z', 'I', 'F', 'T', '6'}
Letters val  {'H', '7', '0', '9', 'L', '5', 'X', 'Y', '3', '2', 'E', 'D', 'M', 'B', 'A', '1', 'O', '4', 'K', 'C', 'G', 'P', '8', 'Z', 'I', 'F', 'T', '6'}
Letters test  {'H', '7', '0', '9', '5', 'X', 'Y', '3', '2', 'E', 'D', 'M', 'B', 'A', '1', 'O', '4', 'K', 'C', 'G', '8', 'P', 'Z', 'I', 'F', 'T', '6'}
Max plate length in train, test and val do match
Letters in train, val and test do match
Letters: 0 1 2 3 4 5 6 7 8 9 A B C D E F G H I K L M O P T X Y Z
START BUILD DATA


100%|████████████████████████████████████████████████████████████████████████████████████████████| 2448/2448 [00:00<00:00, 36605.75it/s]

DATA PREPARED


In [13]:
# # tune
# lr_finder = ocrTextDetector.tune()
#
# # Plot with
# fig = lr_finder.plot(suggest=True)
# fig.show()

In [14]:
ocrTextDetector.train()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


RuntimeError: Lightning can't create new processes if CUDA is already initialized. Did you manually call `torch.cuda.*` functions, have moved the model to the device, or allocated memory on the GPU any other way? Please remove any such calls, or change the selected strategy. You will have to restart the Python kernel.

In [ ]:
ocrTextDetector.save(RESULT_MODEL_PATH, weights_only=False)

In [ ]:
ocrTextDetector.load(RESULT_MODEL_PATH)

In [ ]:
ocrTextDetector.test_acc(verbose=True)

## than train with augumentation

In [ ]:
for i in range(0,1):
    # Train next 2 epochs on augumentated dataset
    ocrTextDetector.epochs += 2

    # prepare with augumentation
    ocrTextDetector.prepare(PATH_TO_DATASET, use_aug=True, num_workers=1, seed=i)

    # Plot with
    #fig = lr_finder.plot(suggest=True)
    #fig.show()
    model = ocrTextDetector.train(seed=i, ckpt_path=RESULT_MODEL_PATH)
    ocrTextDetector.test_acc(verbose=False)
    ocrTextDetector.save(RESULT_MODEL_PATH, weights_only=False)


In [ ]:
# Save only weights results
ocrTextDetector.save(RESULT_MODEL_PATH, weights_only=True)